# P9 – Indicateurs de volume et d'élan (prix 2017 à 2025)
**Contributrice :** Gabrielle · **Date :** 09/11/2026

**Pourquoi :** préparer H5 (« le volume et l'élan annoncent le sens de l'évolution »). Ces indicateurs servent aussi à H7 (variables « quand »).

**Règle de la feuille de route :** la préparation du tour 2 ne touche jamais la cible. Ce notebook ne lit pas le prix de l'année N : il n'utilise que des informations de N-1 et de N-2 (contrôle à la section 2).

**Constat :** pas de nouvelle source. La table de modélisation contient déjà les ventes de N-1 (`nb_ventes_n1`) et le prix de N-1 (`prix_n1`). Pour N-2, on lit les mêmes colonnes sur la ligne de l'année précédente du même territoire.

Pour une ligne (sous-bassin, type de bien, année N) :

| Information | Où on la lit |
|---|---|
| ventes N-1 | `nb_ventes_n1` de la ligne N |
| ventes N-2 | `nb_ventes_n1` de la ligne N-1 du même territoire |
| prix N-1 | `prix_n1` de la ligne N |
| prix N-2 | `prix_n1` de la ligne N-1 du même territoire |


## 0. Chargement
On part de la table **brute** (`masquer=False`). Le masque commun retire les prix 2015 et 2016 (délinquance vide, voir P6). Or la ligne 2016 est nécessaire pour calculer l'élan des prix 2017, premier prix de la période longue. Le masque s'appliquera au moment de modéliser, pas ici.

In [1]:
import numpy as np
import pandas as pd

import commun
from commun import charger_table

ANNEE_DEBUT_LONGUE = 2017             # premiers prix de la période longue (P6)

brute = charger_table(masquer=False)  # toutes les lignes, avant le masque commun
table = charger_table()               # lignes sans case vide : celles des modèles
print("Table brute :", brute.shape, "| après le masque :", table.shape)

Table brute : (28450, 39) | après le masque : (23489, 39)


## 1. Calcul des quatre indicateurs
| Indicateur | Formule | Unité |
|---|---|---|
| `evol_ventes_n2_n1` | ventes N-1 / ventes N-2 − 1 | % |
| `evol_prix_n2_n1` (l'**élan**) | prix N-1 / prix N-2 − 1 | % |
| `rotation_parc` | ventes N-1 / parc total de logements | % par an |
| `rotation_parc_type` | ventes N-1 / parc du même type (appartements ou maisons) | % par an |

**Moins de 20 ventes en N-2 : case vide** (règle à valider en équipe). Une ligne N-1 n'existe dans la table que si le territoire avait au moins 20 ventes en N-1 et en N-2. Sans elle, l'indicateur reste vide. C'est le même seuil que la cible : une médiane sur moins de 20 ventes est trop bruitée pour en tirer une évolution.

**Parc de logements :** `log_nb_logements` est en log10 (recensement 2022), donc parc = 10 puissance `log_nb_logements`. Le parc du même type = parc × `part_apparts` pour les appartements, parc × (100 − `part_apparts`) pour les maisons. Un parc du type nul (aucun appartement) donne une case vide plutôt qu'une division par 0.

In [2]:
CLES = ["sous_bassin_de_vie", "type_bien"]
COLONNES_P9 = ["evol_ventes_n2_n1", "evol_prix_n2_n1", "rotation_parc", "rotation_parc_type"]


def a_plat(table):
    """Index (année, type de bien, sous-bassin) -> colonnes ; `annee` est déjà une colonne."""
    noms = [n for n in table.index.names if n is not None and n not in table.columns]
    return table.reset_index(level=noms).reset_index(drop=True) if noms else table.reset_index(drop=True)


def indicateurs_volume_elan(table):
    """Renvoie une ligne par (année, sous-bassin, type de bien) avec les 4 indicateurs P9.

    `table` : la table de modélisation (colonnes annee, nb_ventes_n1, prix_n1, log_nb_logements,
    part_apparts ; type_bien et sous_bassin_de_vie dans l'index). La colonne `prix` n'est jamais lue.
    """
    t = a_plat(table)
    base = t[["annee"] + CLES + ["nb_ventes_n1", "prix_n1", "log_nb_logements", "part_apparts"]].copy()

    # La ligne N-1 du même territoire, décalée d'un an : ses valeurs « N-1 » sont nos « N-2 »
    precedente = base[["annee"] + CLES + ["nb_ventes_n1", "prix_n1"]].rename(
        columns={"nb_ventes_n1": "ventes_n2", "prix_n1": "prix_n2"})
    precedente["annee"] += 1
    m = base.merge(precedente, on=["annee"] + CLES, how="left", validate="one_to_one")

    m["evol_ventes_n2_n1"] = (m["nb_ventes_n1"] / m["ventes_n2"] - 1) * 100
    m["evol_prix_n2_n1"] = (m["prix_n1"] / m["prix_n2"] - 1) * 100

    parc = 10 ** m["log_nb_logements"]
    part_type = np.where(m["type_bien"] == "appartement",
                         m["part_apparts"], 100 - m["part_apparts"]) / 100
    m["rotation_parc"] = m["nb_ventes_n1"] / parc * 100
    # parc du type nul (aucun appartement) -> case vide plutôt qu'une division par 0
    m["rotation_parc_type"] = m["nb_ventes_n1"] / (parc * np.where(part_type > 0, part_type, np.nan)) * 100

    return m[["annee"] + CLES + ["ventes_n2", "prix_n2"] + COLONNES_P9]


p9 = indicateurs_volume_elan(brute)
assert len(p9) == len(brute), "Le calcul a changé le nombre de lignes"
assert not p9.duplicated(["annee"] + CLES).any(), "Doublons année x sous-bassin x type de bien"
print(p9.shape)
p9.head()

(28450, 9)


,annee,sous_bassin_de_vie,type_bien,ventes_n2,prix_n2,evol_ventes_n2_n1,evol_prix_n2_n1,rotation_parc,rotation_parc_type
0,2015,01004,appartement,NaN,NaN,NaN,NaN,0.236871,0.838041
1,2015,01004,maison,NaN,NaN,NaN,NaN,1.477430,2.059561
2,2015,01033,appartement,NaN,NaN,NaN,NaN,0.635971,1.425470
3,2015,01033,maison,NaN,NaN,NaN,NaN,0.783722,1.415039
4,2015,01034,appartement,NaN,NaN,NaN,NaN,0.309810,1.203719


## 2. La cible n'est jamais lue
Contrôle direct de la règle de la feuille de route : on vide la colonne `prix` (le prix de l'année N) et on recalcule. Les indicateurs doivent être exactement les mêmes.

In [3]:
sans_cible = indicateurs_volume_elan(brute.assign(prix=np.nan))
identique = sans_cible.equals(p9)
print("Indicateurs identiques sans la colonne prix :", identique)
assert identique, "Un indicateur dépend du prix de l'année N : fuite"

Indicateurs identiques sans la colonne prix : True


## 3. Cases vides par année
Part de cases vides (%) par indicateur et par année de prix. D'abord sur la table brute, puis sur les seules lignes gardées par le masque commun (celles des modèles, prix 2017 à 2025).

In [4]:
def tableau_couverture(p9):
    """Part de cases vides par année et par indicateur (en %), et nombre de lignes."""
    return (p9.groupby("annee")[COLONNES_P9].apply(lambda g: g.isna().mean() * 100)
            .round(1).assign(lignes=p9.groupby("annee").size()))


tableau_couverture(p9)

,evol_ventes_n2_n1,evol_prix_n2_n1,rotation_parc,rotation_parc_type,lignes
annee,,,,,
2015,100.0,100.0,0.0,0.0,2460
2016,1.9,1.9,0.0,0.0,2492
2017,1.7,1.7,0.0,0.0,2522
2018,1.8,1.8,0.0,0.0,2554
2019,1.5,1.5,0.0,0.0,2585
2020,2.5,2.5,0.0,0.0,2637
2021,1.2,1.2,0.0,0.0,2659
2022,1.9,1.9,0.0,0.0,2694
2023,0.5,0.5,0.0,0.0,2642


In [5]:
# Lignes des modèles : celles du masque commun, à partir de la période longue
cles_modeles = a_plat(table)[["annee"] + CLES]
p9_modeles = p9.merge(cles_modeles, on=["annee"] + CLES, how="inner")
p9_modeles = p9_modeles[p9_modeles["annee"] >= ANNEE_DEBUT_LONGUE]
couverture_modeles = (p9_modeles.groupby(["annee", "type_bien"])[COLONNES_P9[:2]]
                      .apply(lambda g: g.isna().mean() * 100).round(1).unstack())
couverture_modeles

evol_ventes_n2_n1        evol_prix_n2_n1       
type_bien       appartement maison     appartement maison
annee                                                    
2017                    5.4    0.1             5.4    0.1
2018                    5.4    0.3             5.4    0.3
2019                    4.6    0.1             4.6    0.1
2020                    7.5    0.1             7.5    0.1
2021                    3.6    0.1             3.6    0.1
2022                    5.5    0.1             5.5    0.1
2023                    1.3    0.1             1.3    0.1
2024                    0.4    0.0             0.4    0.0
2025                    2.3    0.1             2.3    0.1

**Lecture**
- **Table brute** : les prix 2015 sont vides à 100 % (pas de ligne 2014 pour lire N-2) ; ils sont de toute façon retirés par le masque commun. À partir de 2016, de 0,1 % à 2,5 % de cases vides par an, seulement pour les deux évolutions. Les deux rotations ne sont jamais vides.
- **Lignes des modèles** (prix 2017 à 2025) : maisons presque complètes (0,3 % de cases vides au plus). Appartements : de 0,4 % à 7,5 % selon l'année (le maximum sur les prix 2020). Ce sont des territoires qui passent tout juste le seuil de 20 ventes en N-1 et en N, mais pas en N-2.
- **Conséquence pour l'équipe** : si les deux évolutions entraient dans le masque commun, elles retireraient jusqu'à 7,5 % des lignes d'appartements à tous les modèles. D'où l'export à part (section 7) et la décision à prendre ensemble (conclusion).

## 4. Vérification à la main (deux territoires)
Pour un appartement (prix 2024) et une maison (prix 2023), on relit les valeurs brutes des lignes N et N-1 et on refait le calcul. Le contrôle s'arrête si le calcul diffère.

In [6]:
def verifier_a_la_main(table, p9, sous_bassin, type_bien, annee):
    """Affiche les valeurs brutes N-1 / N-2 d'un territoire et compare au calcul."""
    t = a_plat(table)
    lignes = t[(t.sous_bassin_de_vie == sous_bassin) & (t.type_bien == type_bien)
               & t.annee.isin([annee - 1, annee])].set_index("annee")
    v_n1, v_n2 = lignes.loc[annee, "nb_ventes_n1"], lignes.loc[annee - 1, "nb_ventes_n1"]
    p_n1, p_n2 = lignes.loc[annee, "prix_n1"], lignes.loc[annee - 1, "prix_n1"]
    calc = p9[(p9.sous_bassin_de_vie == sous_bassin) & (p9.type_bien == type_bien)
              & (p9.annee == annee)].iloc[0]
    print(f"\n{sous_bassin} – {type_bien} – prix {annee}")
    print(f"  ventes N-2 = {v_n2:.0f}, ventes N-1 = {v_n1:.0f} -> évolution attendue "
          f"{(v_n1 / v_n2 - 1) * 100:+.2f} % | calculée {calc.evol_ventes_n2_n1:+.2f} %")
    print(f"  prix N-2 = {p_n2:.0f}, prix N-1 = {p_n1:.0f} -> élan attendu "
          f"{(p_n1 / p_n2 - 1) * 100:+.2f} % | calculé {calc.evol_prix_n2_n1:+.2f} %")
    assert np.isclose(calc.evol_ventes_n2_n1, (v_n1 / v_n2 - 1) * 100)
    assert np.isclose(calc.evol_prix_n2_n1, (p_n1 / p_n2 - 1) * 100)


exemples = p9.dropna(subset=COLONNES_P9)
ex_a = exemples[(exemples.type_bien == "appartement") & (exemples.annee == 2024)].iloc[0]
ex_m = exemples[(exemples.type_bien == "maison") & (exemples.annee == 2023)].iloc[100]
for ex in (ex_a, ex_m):
    verifier_a_la_main(brute, p9, ex.sous_bassin_de_vie, ex.type_bien, int(ex.annee))


01004 – appartement – prix 2024
  ventes N-2 = 130, ventes N-1 = 113 -> évolution attendue -13.08 % | calculée -13.08 %
  prix N-2 = 2039, prix N-1 = 2037 -> élan attendu -0.10 % | calculé -0.10 %

08302 – maison – prix 2023
  ventes N-2 = 93, ventes N-1 = 86 -> évolution attendue -7.53 % | calculée -7.53 %
  prix N-2 = 762, prix N-1 = 847 -> élan attendu +11.15 % | calculé +11.15 %


## 5. Ordres de grandeur
Distribution de chaque indicateur par type de bien, sur la période longue (prix 2017 à 2025).

In [7]:
longue = p9[p9["annee"] >= ANNEE_DEBUT_LONGUE]
(longue.groupby("type_bien")[COLONNES_P9]
 .describe(percentiles=[.05, .5, .95]).T.round(2))

type_bien                 appartement    maison
evol_ventes_n2_n1  count      7185.00  16001.00
                   mean          4.88      2.48
                   std          23.64     17.92
                   min         -64.91    -60.66
                   5%          -27.82    -25.61
                   50%           2.74      1.74
                   95%          46.43     32.80
                   max         238.10    118.18
evol_prix_n2_n1    count      7185.00  16001.00
                   mean          3.61      3.08
                   std           8.93      7.96
                   min         -47.33    -36.51
                   5%           -9.71     -9.01
                   50%           3.25      2.78
                   95%          17.88     15.97
                   max          79.35     92.33
rotation_parc      count      7483.00  16015.00
                   mean          0.87      1.89
                   std           0.68      0.65
                   min           0.06      0.02
                   5%            0.18      0.70
                   50%           0.63      1.91
                   95%           2.30      2.93
                   max           4.09      4.57
rotation_parc_type count      7483.00  16015.00
                   mean          2.28      2.40
                   std           0.96      0.59
                   min           0.42      0.21
                   5%            0.97      1.50
                   50%           2.19      2.37
                   95%           3.89      3.41
                   max          12.91      4.89

**Lecture**
- **Évolution des ventes** : très dispersée. Médiane de +2,7 % (appartements) et +1,7 % (maisons), mais 90 % des valeurs vont de −28 % à +46 % (appartements) et de −26 % à +33 % (maisons). Une partie de cette dispersion est du bruit de petits marchés : passer de 25 à 30 ventes fait déjà +20 %.
- **Élan** : médiane de +3,3 % (appartements) et +2,8 % (maisons) par an ; 90 % des valeurs entre −10 % et +18 %.
- **Rotation du parc du même type** : médiane de 2,2 % des appartements et 2,4 % des maisons vendus chaque année, un ordre de grandeur plausible. Le maximum (12,9 % pour les appartements) vient de territoires où les appartements sont rares : le dénominateur est petit, la valeur grimpe vite.

## 6. Part commune de l'année
Médiane de chaque indicateur d'évolution par année de prix et type de bien. C'est la part commune à tous les territoires (la conjoncture nationale). Contrôle de vraisemblance : la chute des ventes de 2023 doit apparaître sur les prix 2024 (ventes 2023 rapportées à 2022).

Pour H5 : la leçon de H3c est que le gain de l'emploi venait de cette part commune, pas des écarts entre territoires. Il faudra donc séparer chaque indicateur en moyenne de l'année et écart du territoire.

In [8]:
medianes = (longue.groupby(["annee", "type_bien"])[["evol_ventes_n2_n1", "evol_prix_n2_n1"]]
            .median().round(1).unstack())
medianes

evol_ventes_n2_n1        evol_prix_n2_n1       
type_bien       appartement maison     appartement maison
annee                                                    
2017                    7.0    6.5             0.3    0.3
2018                   14.9   13.8             2.1    2.0
2019                    2.3    0.7             1.5    2.1
2020                   13.7   10.6             3.0    2.3
2021                   -0.5    1.1             5.1    4.7
2022                   16.7   16.9             7.6    7.7
2023                   -0.1   -8.1             8.3    9.1
2024                  -18.3  -21.0             2.8    0.8
2025                  -10.8   -7.2            -1.5   -2.8

**Lecture**
- **Le contrôle de vraisemblance passe** : sur les prix 2024, les ventes de 2023 reculent de 18 % (appartements) et de 21 % (maisons) par rapport à 2022 (médiane des territoires), conforme au recul national connu.
- **Le volume tourne avant le prix** : pour les prix 2023, les ventes de maisons reculent déjà (−8 %) alors que l'élan est à son maximum (+9,1 %). L'élan ne retombe qu'ensuite : +0,8 % pour les maisons sur les prix 2024, −2,8 % sur les prix 2025.
- **Ce que ça veut dire pour H5** : c'est exactement l'idée de l'hypothèse, mais vue ici sur la part commune de l'année. Sur les deux années jugées, ces indicateurs risquent de porter surtout du « quand » national, comme l'emploi dans H3c. C'est une piste à tester, pas un résultat.

## 7. Export
Une ligne par année de prix × sous-bassin × type de bien, mêmes clés que la table de modélisation. Les valeurs intermédiaires (ventes et prix N-2) ne sont pas exportées. Le fichier n'est pas fusionné à la table de modélisation : H5 le joint à la table au moment de modéliser, sur les clés.

In [9]:
sortie = commun.DATA_OUT / "p9_volume_elan.parquet"
p9.drop(columns=["ventes_n2", "prix_n2"]).to_parquet(sortie, index=False)
print("-> Exporté :", sortie.name, p9.drop(columns=["ventes_n2", "prix_n2"]).shape)

-> Exporté : p9_volume_elan.parquet (28450, 7)


## Conclusion
- **Quatre indicateurs pour les 28 450 lignes de la table**, mêmes clés (année, sous-bassin, type de bien), exportés dans `p9_volume_elan.parquet`. Pas branchés à la fusion ni à la table de modélisation : H5 les joint au moment de modéliser.
- **La cible n'est jamais lue** (section 2 : mêmes indicateurs sans la colonne `prix`). **Vérification à la main** réussie sur deux territoires (section 4).
- **Couverture** (section 3) : rotations toujours renseignées ; évolutions vides de 0,3 % au plus pour les maisons, de 0,4 % à 7,5 % pour les appartements selon l'année (moins de 20 ventes en N-2).
- **Vraisemblance** (section 6) : la chute des ventes de 2023 apparaît bien (−18 % à −21 % en médiane). Le volume tourne un an avant l'élan.
- **À décider en équipe** :
  1. moins de 20 ventes en N-2 : case vide (choix actuel) ou calcul quand même ;
  2. les cases vides des évolutions : hors du masque commun (ajout à `HORS_MASQUE` ou jointure seulement dans H5), pour ne pas retirer de lignes d'appartements aux autres modèles ;
  3. la bande « stable » de H5 : fixe, ou dépendante du nombre de ventes (le bruit des petits marchés se voit à la section 5).